# FFM_2_9 — landmark XYZ → RFF → MLP → latent

从 `FFM_2_8` 导出的缓存开始。取代 `FFM_2_5_MLP_train.ipynb`。

## 中心问题

> **把每个 landmark 的三维坐标做 Fourier 位置编码，能不能改善从解剖 landmark
> 到 autoencoder latent 的泛化？**

$$p_i=[x_i,y_i,z_i]^\top,\qquad
\gamma(p_i)=[\sin(2\pi Bp_i),\ \cos(2\pi Bp_i)],\qquad
B\in\mathbb R^{M\times3},\ B_{ij}\sim\mathcal N(0,\sigma^2)$$

**注意 `B` 的第二维是 3。** 这是 per-landmark 编码：每个点单独过同一个 `B`，
编码之后才拼接。和把 57 维拍平向量整体投影是**两件不同的事**，后者在这里只作为消融。

**文献里没有一篇这么做。** RFF 在 NIIRF / SuDaField / GraphNF-SCA / Xiao 里加在**方向**上，
在 Niu 和本项目的 decoder 里加在**频率**上，在 Martinez 里加在**场坐标**上。
把 RFF 加在一组 3-D landmark 上是 Craig 自己的提议，**没有参考实现，是潜在的贡献点**。

## σ 的量级——最容易让实验因为一个常数而失败的地方

坐标跨度 ±143 mm，centroid size 510.8 ± 21.3 mm。换成**米**之后典型 ‖p‖ ≈ 0.1 m。

要分辨 **5–10 mm** 的耳廓特征，需要空间周期 0.005–0.01 m，即 **‖b‖ ≈ 100–200 m⁻¹**。

**所以米制下 σ 的量级是 50–200，不是 1.0。** 用 σ=1 的话整个头上相位只变 ~0.6 弧度，
sin/cos 在那个范围内几乎是线性的——**RFF 退化成一个随机线性投影，什么都不做**，
然后你会得到"RFF 没用"的假结论。

Cell 2 会打印 `median ‖b‖` 和对应的 **λ = 1/‖b‖（换算成 mm）**，一眼看出这组 σ 在分辨多大的结构。

## 维度会爆

$D_\mathrm{RFF} = P\times 2M$，P=19：

| M | 维度 | vs N_train=37 |
|---:|---:|---|
| 4 | 152 | 4.1× |
| 8 | 304 | 8.2× |
| 16 | 608 | 16× |
| 32 | 1216 | 33× |
| ~~512~~ | ~~19456~~ | ~~526×~~ 不要用 |

所以 `RFF_M_GRID` 从小开始，用内层 CV 选，**绝不用 test 选**。

## 修掉的泄漏

`FFM_2_4` 的 ridge 把 latent PCA 在**全部被试**上拟合、之后才切 CV 折。本 notebook
的 PCA / 均值 / 标准化统计量**全部 fold-local**，只用训练折。

## 已知的上游结果（2_8）

dCor²ᵤ = 0.0710，p = 0.0020，N = 61；负对照 −0.008 / −0.031，都不显著。
分带：**DC–0.5k 0.140 / 0.5–6k 0.128 / 6–12k 0.063 / 12–15.8k 0.073**。
信号最强在头尺寸主导的低频，**最弱恰好在耳廓带**。

In [1]:
# ==========================================================================
# Cell 0 — 载入 FFM_2_8 的缓存
# ==========================================================================
import os, sys, json, time, hashlib, itertools, warnings
warnings.filterwarnings('ignore')
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
else:
    DRIVE = os.environ.get('ELEC5021_DRIVE', os.path.expanduser('~/Documents/26S2/ELEC5021'))

SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
CACHE    = f'{SAVE_DIR}/landmark2z_v28_cache.npz'
assert os.path.exists(CACHE), f'找不到 {CACHE} —— 先跑 FFM_2_8 的最后一格'

d = np.load(CACHE, allow_pickle=True)
XYZ_SUB  = d['xyz_sub'].astype(np.float64)        # (N, P, 3) mm，左耳坐标系，未中心化
XYZ_EAR  = d['xyz_ear'].astype(np.float64)        # (2N, P, 3)
Z_SUB    = d['Z_sub'].astype(np.float64)          # (N, 5952)
Z_FD_SUB = d['Z_fd_sub'].astype(np.float64)       # (N, 93, 64)
SYM, AE_PART = d['sym'], d['ae_part']
N, P, C = XYZ_SUB.shape
assert C == 3, f'最后一维必须是 3，实际 {C}'
n_bins, LATENT_DIM = int(d['n_bins']), int(d['latent_dim'])
SUBJ_HASH = str(d['subject_hash'])
BAND_RES  = json.loads(str(d['band_res']))

print(f'缓存 <- {CACHE}')
print(f'  xyz_sub {XYZ_SUB.shape}  单位 {d["coord_unit"]}  frame: {d["coord_frame"]}')
print(f'  Z_sub   {Z_SUB.shape}   ({n_bins} bins x {LATENT_DIM})')
print(f'  N={N} 人 | P={P} 点 (point_set={d["point_set"]}) | hash {SUBJ_HASH}')
print(f'  ckpt {d["ckpt"]} | side {float(d["side_frac"]):.1%} 投影={bool(d["side_projected"])}')
from collections import Counter
print(f'  AE 分区 {dict(Counter(AE_PART))}')
print(f'\n2_8 的 dCor²ᵤ = {float(d["dcor_obs"]):+.4f}, p = {float(d["dcor_p"]):.4f}')
print('  分带:', {k: f'{v[0]:+.4f}(p={v[1]:.3f})' for k, v in BAND_RES.items()})

Mounted at /content/drive
缓存 <- /content/drive/MyDrive/HRTF_Checkpoints/landmark2z_v28_cache.npz
  xyz_sub (61, 19, 3)  单位 mm  frame: left-ear, SYMARE common frame, uncentred
  Z_sub   (61, 5952)   (93 bins x 64)
  N=61 人 | P=19 点 (point_set=ipsi) | hash 5d151b755ad8
  ckpt FFM26_best.pth | side 0.2% 投影=False
  AE 分区 {np.str_('train'): 37, np.str_('test'): 15, np.str_('val'): 9}

2_8 的 dCor²ᵤ = +0.0710, p = 0.0020
  分带: {'0-500': '+0.1399(p=0.005)', '500-6000': '+0.1282(p=0.005)', '6000-12000': '+0.0631(p=0.015)', '12000-15849': '+0.0725(p=0.005)'}


In [2]:
# ==========================================================================
# Cell 1 — ★ 配置（唯一要动的地方）
# ==========================================================================
CFG = dict(
    # ---------------- 输入编码 ----------------
    INPUT_ENCODING = 'rff_per_landmark',   # 'raw' | 'rff_per_landmark' | 'rff_global'
    RFF_INCLUDE_RAW = False,               # True 时把原始坐标和 RFF 拼在一起

    # ---------------- 坐标预处理 ----------------
    COORD_CENTER = 'subject_centroid',     # 'none' | 'subject_centroid'
    COORD_UNIT   = 'm',                    # 'mm' | 'm'   ★ 决定 σ 的量级
    # ★ 绝不做 per-subject 尺度归一化 —— 头/耳尺寸是真信号

    # ---------------- RFF 超参（内层 CV 选，绝不用 test）----------------
    RFF_M_GRID     = [4, 8, 16, 32],
    RFF_SIGMA_GRID = [20., 50., 100., 200., 500.],   # 米制。mm 制要除以 1000
    RFF_M_GLOBAL   = [64, 128, 256],                 # global 消融用

    # ---------------- 目标 ----------------
    K_VAR = 0.95,          # fold-local PCA 保留多少 latent 方差

    # ---------------- MLP（三种编码共用，保证公平）----------------
    HIDDEN = 128, N_LAYERS = 2, DROPOUT = 0.2,
    LR = 1e-3, WEIGHT_DECAY = 1e-2, BATCH = 8, EPOCHS = 500,

    # ---------------- 评估 ----------------
    N_FOLDS = 10, N_SEEDS = 3, SEED = 0,
)
STOP_RULE = 0.10          # 事先约定：CV-R² 低于此就停，不加容量

if CFG['COORD_UNIT'] == 'mm':
    CFG['RFF_SIGMA_GRID'] = [s / 1000 for s in CFG['RFF_SIGMA_GRID']]
    print('⚠️ COORD_UNIT=mm —— σ 网格已自动除以 1000 以保持同样的空间频率')

print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=str))
D_rff = P * 2 * np.array(CFG['RFF_M_GRID'])
print(f'\nper-landmark RFF 维度 D = P·2M = {list(D_rff)}  (P={P})')
print(f'raw 维度 = {P*3}')

{
 "INPUT_ENCODING": "rff_per_landmark",
 "RFF_INCLUDE_RAW": false,
 "COORD_CENTER": "subject_centroid",
 "COORD_UNIT": "m",
 "RFF_M_GRID": [
  4,
  8,
  16,
  32
 ],
 "RFF_SIGMA_GRID": [
  20.0,
  50.0,
  100.0,
  200.0,
  500.0
 ],
 "RFF_M_GLOBAL": [
  64,
  128,
  256
 ],
 "K_VAR": 0.95,
 "HIDDEN": 128,
 "N_LAYERS": 2,
 "DROPOUT": 0.2,
 "LR": 0.001,
 "WEIGHT_DECAY": 0.01,
 "BATCH": 8,
 "EPOCHS": 500,
 "N_FOLDS": 10,
 "N_SEEDS": 3,
 "SEED": 0
}

per-landmark RFF 维度 D = P·2M = [np.int64(152), np.int64(304), np.int64(608), np.int64(1216)]  (P=19)
raw 维度 = 57


In [3]:
# ==========================================================================
# Cell 2 — 坐标预处理 · Fourier 编码器
#
# B 是【固定】随机特征，不是可训练参数（§5）。用 nn.Module + register_buffer，
# 这样 B 会随 state_dict 一起存，resume 不可能重新生成一组不同的基。
# ==========================================================================
import torch, torch.nn as nn

def preprocess_coords(xyz, center, unit):
    """xyz: (N, P, 3) mm -> 预处理后的 (N, P, 3)。★ 不做尺度归一化。"""
    X = xyz.copy()
    if center == 'subject_centroid':
        X = X - X.mean(1, keepdims=True)
    elif center != 'none':
        raise ValueError(center)
    if unit == 'm':
        X = X / 1000.0
    elif unit != 'mm':
        raise ValueError(unit)
    return X


class PerLandmarkFourierFeatures(nn.Module):
    """γ(p_i) = [sin(2πB p_i), cos(2πB p_i)]，B ∈ R^(M×3)，所有 landmark 共享。
    (N, P, 3) -> (N, P·2M)。landmark 顺序保持不变，不做池化。"""
    def __init__(self, M, sigma, seed, include_raw=False):
        super().__init__()
        self.M, self.sigma, self.seed, self.include_raw = M, sigma, seed, include_raw
        B = np.random.default_rng(seed).normal(0.0, sigma, size=(M, 3))
        self.register_buffer('B', torch.tensor(B, dtype=torch.float64))

    def out_dim(self, P):
        return P * (2 * self.M + (3 if self.include_raw else 0))

    def forward(self, xyz):                      # xyz: (N, P, 3) torch
        assert xyz.ndim == 3 and xyz.shape[-1] == 3
        N_, P_, _ = xyz.shape
        phase = 2 * np.pi * (xyz @ self.B.T)     # (N, P, M)
        assert phase.shape == (N_, P_, self.M)
        f = torch.cat([torch.sin(phase), torch.cos(phase)], dim=-1)   # (N, P, 2M)
        assert f.shape == (N_, P_, 2 * self.M)
        if self.include_raw:
            f = torch.cat([xyz, f], dim=-1)      # (N, P, 3+2M)
        return f.reshape(N_, -1)                 # (N, P·(2M[+3]))


class GlobalFourierFeatures(nn.Module):
    """消融：先拍平成 (N, 3P)，再整体投影。B ∈ R^(M×3P)。"""
    def __init__(self, M, sigma, D_in, seed, include_raw=False):
        super().__init__()
        self.M, self.sigma, self.seed, self.include_raw = M, sigma, seed, include_raw
        B = np.random.default_rng(seed + 10_000).normal(0.0, sigma, size=(M, D_in))
        self.register_buffer('B', torch.tensor(B, dtype=torch.float64))

    def out_dim(self, D_in):
        return 2 * self.M + (D_in if self.include_raw else 0)

    def forward(self, xyz):
        X = xyz.reshape(len(xyz), -1)            # (N, 3P)
        phase = 2 * np.pi * (X @ self.B.T)
        f = torch.cat([torch.sin(phase), torch.cos(phase)], dim=-1)
        return torch.cat([X, f], dim=-1) if self.include_raw else f


def make_encoder(mode, M, sigma, seed, include_raw, P_):
    if mode == 'raw':      return None
    if mode == 'rff_per_landmark':
        return PerLandmarkFourierFeatures(M, sigma, seed, include_raw)
    if mode == 'rff_global':
        return GlobalFourierFeatures(M, sigma, P_ * 3, seed, include_raw)
    raise ValueError(mode)

def encode(xyz_np, mode, M=None, sigma=None, seed=0, include_raw=False):
    """统一入口，【总是】返回 numpy (N, D)。B 固定，不参与梯度。"""
    X = torch.tensor(xyz_np, dtype=torch.float64)
    enc = make_encoder(mode, M, sigma, seed, include_raw, X.shape[1])
    if enc is None:
        return X.reshape(len(X), -1).numpy()
    with torch.no_grad():
        return enc(X).numpy()


# ---------------- 预处理 + σ 的物理尺度诊断 ----------------
XYZ = preprocess_coords(XYZ_SUB, CFG['COORD_CENTER'], CFG['COORD_UNIT'])
u = CFG['COORD_UNIT']
print(f"预处理: center={CFG['COORD_CENTER']} unit={u}  ★不做尺度归一化")
print(f'  坐标范围 {XYZ.min():+.4f} .. {XYZ.max():+.4f} {u} | '
      f'典型 ‖p‖ {np.linalg.norm(XYZ, axis=-1).mean():.4f} {u}')

print(f"\n{'σ':>8}{'median ‖b‖':>14}{'10%':>10}{'90%':>10}{'λ=1/‖b‖ [mm]':>16}")
for s in CFG['RFF_SIGMA_GRID']:
    B = np.random.default_rng(CFG['SEED']).normal(0, s, size=(256, 3))
    nb_ = np.linalg.norm(B, axis=1)
    lam_mm = 1.0 / np.median(nb_) * (1000 if u == 'm' else 1)
    print(f'{s:>8.4g}{np.median(nb_):>14.2f}{np.quantile(nb_,.1):>10.2f}'
          f'{np.quantile(nb_,.9):>10.2f}{lam_mm:>16.2f}')
print('★ λ 就是这组 σ 能分辨的结构尺度。耳廓特征在 5–10 mm —— 挑 λ 落在那附近的 σ。')
print('  λ 远大于 100 mm 的 σ 只能看到头尺寸；远小于 1 mm 的只在拟合噪声。')

预处理: center=subject_centroid unit=m  ★不做尺度归一化
  坐标范围 -0.1251 .. +0.1399 m | 典型 ‖p‖ 0.0616 m

       σ    median ‖b‖       10%       90%    λ=1/‖b‖ [mm]
      20         30.91     16.19     49.25           32.35
      50         77.28     40.48    123.12           12.94
     100        154.56     80.96    246.23            6.47
     200        309.12    161.92    492.46            3.23
     500        772.81    404.79   1231.15            1.29
★ λ 就是这组 σ 能分辨的结构尺度。耳廓特征在 5–10 mm —— 挑 λ 落在那附近的 σ。
  λ 远大于 100 mm 的 σ 只能看到头尺寸；远小于 1 mm 的只在拟合噪声。


In [ ]:
# ==========================================================================
# Cell 3 — 划分 · fold-local 目标构造
#
# ★ PCA / 均值 / 标准化统计量【全部只用训练折】。FFM_2_4 的泄漏就在这。
# ==========================================================================
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

I_TR = np.where(AE_PART == 'train')[0]
I_VA = np.where(AE_PART == 'val')[0]
I_TE = np.where(AE_PART == 'test')[0]
S_TR, S_VA, S_TE = set(I_TR.tolist()), set(I_VA.tolist()), set(I_TE.tolist())
assert not (S_TR & S_VA), 'train/val 有重叠'
assert not ((S_TR | S_VA) & S_TE), '★ train/val 碰到了 test'
assert S_TR | S_VA | S_TE == set(range(N)), '划分没有覆盖全部受试者'
DEV = np.concatenate([I_TR, I_VA])          # 开发集：选超参用，test 一律不碰
print(f'train {len(I_TR)} | val {len(I_VA)} | test {len(I_TE)} 人  '
      f'（AE 的划分，直接沿用）')
print(f'开发集 = train+val = {len(DEV)} 人；所有超参在这里面用内层 CV 选')


def make_target(idx_fit, k_var=None):
    """只用 idx_fit 拟合 latent 的 PCA，返回 (投影函数, 逆投影函数, K)"""
    k_var = k_var or CFG['K_VAR']
    mu = Z_SUB[idx_fit].mean(0)
    U, S, Vt = np.linalg.svd(Z_SUB[idx_fit] - mu, full_matrices=False)
    K = int(np.searchsorted(np.cumsum(S**2) / (S**2).sum(), k_var) + 1)
    Pm = Vt[:K]                                   # (K, 5952)
    return (lambda Zr: (Zr - mu) @ Pm.T), (lambda Y: Y @ Pm + mu), K


def r2_score(Y_true, Y_pred, Y_train_mean):
    """基线 = 训练折的目标均值。0 = 和直接猜均值一样好，负数 = 更差。"""
    ss_res = ((Y_true - Y_pred) ** 2).sum()
    ss_tot = ((Y_true - Y_train_mean) ** 2).sum()
    return 1 - ss_res / ss_tot


# ---------------------------------------------------------------- 完整性哨兵
# ★ 负对照会临时打乱 Z_SUB。如果那一格在还原前被中断，后面每一格都会静默
#   产出垃圾且不报错。指纹 + assert 让它响亮地失败。
Z_FINGERPRINT = hashlib.md5(np.ascontiguousarray(Z_SUB)).hexdigest()[:12]

def assert_Z_clean():
    h = hashlib.md5(np.ascontiguousarray(Z_SUB)).hexdigest()[:12]
    assert h == Z_FINGERPRINT, (
        f'★ Z_SUB 已被改动 ({h} != {Z_FINGERPRINT})。'
        '最可能的原因：负对照那一格在还原前被中断。请重跑 Cell 0。')


def standardize(X_fit, X_apply):
    mu, sd = X_fit.mean(0), X_fit.std(0) + 1e-12
    return (X_fit - mu) / sd, [(x - mu) / sd for x in X_apply]

_, _, K_dev = make_target(DEV)
print(f'\n开发集上 {CFG["K_VAR"]:.0%} 方差需 K = {K_dev} 维'
      f'（每折会各自重算，这里只是参考）')

In [ ]:
# ==========================================================================
# Cell 4 — Ridge：三种编码的对照（闭式解，无 SGD）
# ==========================================================================
ALPHAS = [1e-1, 1., 10., 1e2, 1e3, 1e4]

assert_Z_clean()          # ★ 开跑前确认 Z_SUB 没被上一格的负对照污染

def build_X(mode, M, sigma, seed, include_raw):
    return encode(XYZ, mode, M, sigma, seed, include_raw)

def cv_ridge(mode, M, sigma, alpha, seed, include_raw, idx, folds=None):
    """在 idx 上做 K 折 CV。每折：fold-local PCA + fold-local 标准化。"""
    X_all = build_X(mode, M, sigma, seed, include_raw)
    kf = KFold(folds or CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])
    scores = []
    for a, b in kf.split(idx):
        i_tr, i_te = idx[a], idx[b]
        fwd, _, _ = make_target(i_tr)                     # ★ fold-local
        Y_tr, Y_te = fwd(Z_SUB[i_tr]), fwd(Z_SUB[i_te])
        Xtr, (Xte,) = standardize(X_all[i_tr], [X_all[i_te]])
        pred = Ridge(alpha=alpha).fit(Xtr, Y_tr).predict(Xte)
        scores.append(r2_score(Y_te, pred, Y_tr.mean(0)))
    scores = np.asarray(scores)
    # ★ 返回逐折向量：各行用的是同一个 KFold(random_state)，折是一一对应的，
    #   所以行与行之间可以做【配对】检验，比「SEM 相加」更有力也更正确。
    return float(scores.mean()), float(scores.std() / np.sqrt(len(scores))), scores


def select_and_score(mode, include_raw, tag):
    """内层 CV 选超参（只在开发集），返回最优配置和它的 CV-R²"""
    grid = ([(None, None)] if mode == 'raw' else
            [(m, s) for m in (CFG['RFF_M_GRID'] if mode == 'rff_per_landmark'
                              else CFG['RFF_M_GLOBAL'])
                    for s in CFG['RFF_SIGMA_GRID']])
    best = (-9e9, None)
    for M, s in grid:
        for al in ALPHAS:
            sc, _, _ = cv_ridge(mode, M, s, al, CFG['SEED'], include_raw, DEV, folds=5)
            if sc > best[0]: best = (sc, dict(M=M, sigma=s, alpha=al))
    cfg = best[1]
    sc, sem, folds = cv_ridge(mode, cfg['M'], cfg['sigma'], cfg['alpha'],
                              CFG['SEED'], include_raw, DEV, folds=CFG['N_FOLDS'])
    D = build_X(mode, cfg['M'], cfg['sigma'], CFG['SEED'], include_raw).shape[1]
    print(f'  {tag:<32}CV-R² {sc:+.4f} ± {sem:.4f} | D={D:>5} | '
          f"M={cfg['M']} σ={cfg['sigma']} α={cfg['alpha']:g}")
    return dict(tag=tag, mode=mode, include_raw=include_raw, D=D, cv=sc, sem=sem,
                folds=folds, **cfg)

print(f'Ridge —— 内层 5 折选超参，外层 {CFG["N_FOLDS"]} 折报分，全部在开发集 {len(DEV)} 人上\n')
RIDGE = [
    select_and_score('raw',               False, 'Raw XYZ + Ridge'),
    select_and_score('rff_per_landmark',  False, 'Per-landmark RFF + Ridge'),
    select_and_score('rff_per_landmark',  True,  'Raw + Per-landmark RFF + Ridge'),
    select_and_score('rff_global',        False, 'Global RFF + Ridge'),
]

# =========================================================================
# 负对照 —— 三个层次，缺一不可
#
# ★ 修正 1：旧版把 α 写死成 1.0，而正式四行选出来的是 1e3~1e4。α=1 在
#   D=57 / 每折 41 人上几乎没有正则，打乱之前它就已经是 −4 了。对照必须走
#   【同一条超参选择路径】，否则它控制不了任何东西。
# ★ 修正 2：try/finally 护住 Z_SUB —— 中断也保证还原。
# ★ 修正 3：一次 shuffle 是一次抽样。ridge 是闭式解，跑一整个零分布几乎不花钱。
#
# ★ 零点在哪：r2_score 的分母基线是【训练折的目标均值】，不是测试折自己的
#   均值。所以「常数预测器」的 R² 恰好是 0，打乱后的期望值也应当略低于 0
#   （ridge 收缩得不完全）。落在 −0.10 ~ +0.02 算干净。
# =========================================================================
N_PERM = 20

# ---- (a) 常数预测器：校准零点。这一条检验的是 r2_score 的口径本身 ----
_ytr = np.random.default_rng(0).normal(size=(20, 4))
_yte = np.random.default_rng(1).normal(size=(7, 4))
_const = r2_score(_yte, np.tile(_ytr.mean(0), (len(_yte), 1)), _ytr.mean(0))
assert abs(_const) < 1e-12, f'★ r2_score 口径不对：常数预测器应当给 0，实际 {_const}'
print(f'  常数预测器 R² = {_const:+.1e}  ← 零点校准通过（基线 = 训练折均值）')

# ---- (b) 打乱目标，跑完整的零分布，每次都自己重选 α ----
def _null_once(perm_seed):
    """打乱 Z_SUB 的行，走和正式行完全相同的路径（内层选 α + 外层报分）。"""
    global Z_SUB
    _keep = Z_SUB
    try:
        Z_SUB = Z_SUB[np.random.default_rng(perm_seed).permutation(N)]
        best = (-9e9, None)
        for al in ALPHAS:                                   # ★ 和正式行同一个网格
            sc, _, _ = cv_ridge('raw', None, None, al, CFG['SEED'], False, DEV, folds=5)
            if sc > best[0]: best = (sc, al)
        sc, _, folds = cv_ridge('raw', None, None, best[1],
                                CFG['SEED'], False, DEV, folds=CFG['N_FOLDS'])
        return sc, best[1]
    finally:
        Z_SUB = _keep                                       # ★ 中断也还原

print(f'\n  负对照：{N_PERM} 次 permutation，每次自己重选 α …')
NULL = np.array([_null_once(1000 + k)[0] for k in range(N_PERM)])
assert_Z_clean()                                            # ★ 确认真的还原了

lo, hi = np.quantile(NULL, [0.025, 0.975])
print(f'  零分布  中位数 {np.median(NULL):+.4f} | 95% 区间 [{lo:+.4f}, {hi:+.4f}] '
      f'| 最差 {NULL.min():+.4f}')
_ok = abs(np.median(NULL)) < 0.10
print(f'  {"[OK] 管道干净" if _ok else "[FAIL] 零分布没落在 0 附近 —— 先别看上面的表"}'
      f'（判据：|中位数| < 0.10）')

# ---- (c) 把正式的 raw 行放到零分布上，给一个经验 p ----
_raw = [r for r in RIDGE if r['mode'] == 'raw'][0]['cv']
_p = (1 + (NULL >= _raw).sum()) / (N_PERM + 1)
print(f'\n  Raw XYZ 观测值 {_raw:+.4f} 对零分布的经验 p = {_p:.4f} '
      f'(N_PERM={N_PERM}，p 的下限是 {1/(N_PERM+1):.4f})')
print('  ★ 这个 p 回答的是「有没有信号」，不是「够不够用」。够不够用看 CV-R² 的绝对值。')


In [ ]:
# ==========================================================================
# Cell 4b — 边界诊断（★ 这不是重新选型，头条数字不变）
#
# 为什么要跑：正式表里 per-landmark RFF 选中 σ=20（网格【下界】）、α=1e4
# （网格【上界】）。两个解都撞在边界上，那个配置本身没法解释 —— 你不知道
# CV 是「找到了 σ=20」还是「想要更小的 σ 但网格不给」。
#
# 这一格唯一的目的是确认那个解释：CV 想要的是【最线性】的那一端。
#   σ 小 -> 波长 λ=1/σ 长 -> sin(2πBx)≈2πBx、cos≈1 -> RFF 退化成
#   xyz 的随机线性投影 + 常数。
#
# ★ 停止规则已经触发。就算延格之后出现更高的 CV，头条数字仍然是原网格上的
#   那个；更高的分数只是加强「这个映射是线性的」，不能把方法救回来。
#   报告时写成 diagnostic，和主表分开。
# ==========================================================================
RUN_DIAG = True

if RUN_DIAG:
    assert_Z_clean()
    SIG_EXT = [2., 5., 10.] + list(CFG['RFF_SIGMA_GRID']) + [1000.]
    ALP_EXT = ALPHAS + [1e5, 1e6]
    print(f'σ 延到 {SIG_EXT}\nα 延到 {[f"{a:g}" for a in ALP_EXT]}\n')
    print(f"{'σ':>8}{'λ [mm]':>10}{'best CV-R²':>14}{'M':>5}{'α':>10}")
    print('-' * 47)
    surf = []
    for sg in SIG_EXT:
        b = (-9e9, None, None)
        for M in CFG['RFF_M_GRID']:
            for al in ALP_EXT:
                sc, _, _ = cv_ridge('rff_per_landmark', M, sg, al,
                                    CFG['SEED'], False, DEV, folds=5)
                if sc > b[0]: b = (sc, M, al)
        lam = 1000.0 / sg if CFG['COORD_UNIT'] == 'm' else 1.0 / sg
        surf.append((sg, b[0], b[1], b[2]))
        print(f'{sg:>8.4g}{lam:>10.1f}{b[0]:>+14.4f}{b[1]:>5}{b[2]:>10.4g}')
    print('-' * 47)

    sg_star = max(surf, key=lambda t: t[1])
    _lam = (1000.0 if CFG['COORD_UNIT'] == 'm' else 1.0) / sg_star[0]
    print(f'\n延格后最好: σ={sg_star[0]:g} (λ={_lam:.0f} mm) CV-R² {sg_star[1]:+.4f}')
    _edge = sg_star[0] in (SIG_EXT[0], SIG_EXT[-1])
    print('★ 仍然撞边界，解释还没定下来。' if _edge else
          '★ 最优 σ 落在网格内部，边界效应排除。')
    if sg_star[0] <= 10.:
        print('★ CV 选了最线性的一端 —— 与「RFF 的增益来自条件数而非表达能力」一致。')
    print(f'\n★ 头条数字不变，仍是原网格上的 {max(r["cv"] for r in RIDGE):+.4f}。'
          '\n  本格为 diagnostic，单独成表，不进主结果。')
else:
    print('RUN_DIAG = False —— 跳过边界诊断。')


In [ ]:
# ==========================================================================
# Cell 5 — MLP：三种编码用【完全相同】的网络和训练过程
#
# §8：为了公平，隐层数/宽度/激活/dropout/优化器/lr/weight decay/epochs 全部固定。
# 只有输入维度不同 —— 参数量因此不同，这是可以接受的，但必须报出来。
# ==========================================================================
assert_Z_clean()          # ★ 同上


def run_mlp_fold(X_all, i_tr, i_te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    fwd, _, K = make_target(i_tr)                          # ★ fold-local
    Y_tr, Y_te = fwd(Z_SUB[i_tr]), fwd(Z_SUB[i_te])
    Xtr, (Xte,) = standardize(X_all[i_tr], [X_all[i_te]])
    Xt = torch.tensor(Xtr, dtype=torch.float32)
    Yt = torch.tensor(Y_tr, dtype=torch.float32)
    Xv = torch.tensor(Xte, dtype=torch.float32)

    layers, dcur = [], Xt.shape[1]
    for _ in range(CFG['N_LAYERS']):
        layers += [nn.Linear(dcur, CFG['HIDDEN']), nn.GELU(), nn.Dropout(CFG['DROPOUT'])]
        dcur = CFG['HIDDEN']
    layers += [nn.Linear(dcur, K)]
    net = nn.Sequential(*layers)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG['LR'], weight_decay=CFG['WEIGHT_DECAY'])
    ds = torch.utils.data.TensorDataset(Xt, Yt)
    dl = torch.utils.data.DataLoader(ds, batch_size=CFG['BATCH'], shuffle=True)
    for _ in range(CFG['EPOCHS']):
        net.train()
        for xb, yb in dl:
            opt.zero_grad(); nn.functional.mse_loss(net(xb), yb).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pr_te = net(Xv).numpy(); pr_tr = net(Xt).numpy()
    n_par = sum(p.numel() for p in net.parameters())
    return (r2_score(Y_te, pr_te, Y_tr.mean(0)),
            r2_score(Y_tr, pr_tr, Y_tr.mean(0)), n_par, K)


def mlp_experiment(cfg_row):
    """用 ridge 选出来的 M/σ（同一组编码），跑 N_SEEDS 个种子 x N_FOLDS 折"""
    X_all = build_X(cfg_row['mode'], cfg_row['M'], cfg_row['sigma'],
                    CFG['SEED'], cfg_row['include_raw'])
    te_all, tr_all, npar = [], [], 0
    for sd in range(CFG['N_SEEDS']):
        kf = KFold(CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'] + sd)
        for a, b in kf.split(DEV):
            te, tr, npar, K = run_mlp_fold(X_all, DEV[a], DEV[b], seed=sd)
            te_all.append(te); tr_all.append(tr)
    te_all, tr_all = np.array(te_all), np.array(tr_all)
    print(f"  {cfg_row['tag'].replace('Ridge','MLP'):<32}"
          f'CV-R² {te_all.mean():+.4f} ± {te_all.std()/np.sqrt(len(te_all)):.4f} | '
          f'train {tr_all.mean():+.4f} | D={X_all.shape[1]:>5} | '
          f'{npar/1e3:.0f}k 参数 = {npar/len(I_TR):.0f}/训练被试')
    row = {k: v for k, v in cfg_row.items() if k != 'folds'}
    return dict(**row, mlp_cv=float(te_all.mean()),
                mlp_sem=float(te_all.std()/np.sqrt(len(te_all))),
                mlp_train=float(tr_all.mean()), n_par=int(npar),
                folds=cfg_row['folds'],        # ridge 的逐折分数，原样带着
                mlp_folds=te_all)              # ★ MLP 的逐折分数，配对检验用

_n_fit = 4 * CFG['N_SEEDS'] * CFG['N_FOLDS']
print(f'MLP —— {CFG["N_SEEDS"]} 个种子 x {CFG["N_FOLDS"]} 折 x 4 种编码 = {_n_fit} 次训练')
print(f'⚠️ 每次 {CFG["EPOCHS"]} epoch，CPU 上约 5–15 s -> 总计约 {_n_fit*10/60:.0f} 分钟。'
      '先把 N_SEEDS 设 1 试通。')
print(f'固定: {CFG["N_LAYERS"]}x{CFG["HIDDEN"]} GELU dropout={CFG["DROPOUT"]} '
      f'AdamW lr={CFG["LR"]} wd={CFG["WEIGHT_DECAY"]} {CFG["EPOCHS"]} epochs\n')
MLP = [mlp_experiment(r) for r in RIDGE]

In [ ]:
# ==========================================================================
# Cell 5 — 对比表
# ==========================================================================
def _fmt(r, cv, sem, tr, tag):
    sg = f"{r['sigma']:.0f}" if r['sigma'] else '—'
    tr_s = f'{tr:+.4f}' if tr is not None else '—'
    print(f"{tag:<34}{cv:>+10.4f}{sem:>8.4f}{tr_s:>10}{r['D']:>7}{str(r['M']):>5}{sg:>8}")

print(f"{'Model':<34}{'CV-R²':>10}{'±SEM':>8}{'train R²':>10}{'D':>7}{'M':>5}{'σ':>8}")
print('-' * 82)
for r in RIDGE: _fmt(r, r['cv'], r['sem'], None, r['tag'])
print('-' * 82)
for r in MLP:   _fmt(r, r['mlp_cv'], r['mlp_sem'], r['mlp_train'],
                     r['tag'].replace('Ridge', 'MLP'))
print('-' * 82)

BEST = max(RIDGE + MLP, key=lambda r: r.get('mlp_cv', r.get('cv', -9)))
best_cv = BEST.get('mlp_cv', BEST.get('cv'))


# -------------------------------------------------------------- 配对检验
def compare(a, b, key, tag_a, tag_b):
    """a - b 的【配对】检验。各行用同一个 KFold，折一一对应。"""
    fa, fb = np.asarray(a[key]), np.asarray(b[key])
    assert fa.shape == fb.shape, '折数对不上，不能配对'
    d = fa - fb
    sem_d = float(d.std(ddof=1) / np.sqrt(len(d)))
    ma = float(fa.mean()); mb = float(fb.mean())
    sem_sum = float(fa.std(ddof=1)/np.sqrt(len(fa)) + fb.std(ddof=1)/np.sqrt(len(fb)))
    print(f'  {tag_a} {ma:+.4f}  vs  {tag_b} {mb:+.4f}   差 {d.mean():+.4f}')
    print(f'    配对 SEM {sem_d:.4f} -> {abs(d.mean())/max(sem_d,1e-12):.1f}x'
          f'   |   两 SEM 之和 {sem_sum:.4f} -> {abs(d.mean())/max(sem_sum,1e-12):.1f}x')
    verdict = (d.mean() > sem_sum)
    print(f'    判据（差值 > 两 SEM 之和）：{"通过" if verdict else "★ 不通过"}')
    # ---- 护栏：两个都在 0 以下时，比较没有意义 ----
    if ma < 0 and mb < 0:
        print('    ⚠️ 两行都低于 0 —— 都比「直接猜训练折均值」还差。')
        print('       「比无用更不无用」不是结果。这个差值只能说明数值条件不同，')
        print('       不能写成 “RFF improves over raw coordinates by X R²”。')
    return verdict


raw_r = [r for r in RIDGE if r['mode'] == 'raw'][0]
rff_r = [r for r in RIDGE if r['mode'] == 'rff_per_landmark' and not r['include_raw']][0]
print('\n中心问题：per-landmark RFF 比 raw XYZ 好吗？')
print(' [Ridge]')
compare(rff_r, raw_r, 'folds', 'RFF', 'raw')

if MLP:
    raw_m = [r for r in MLP if r['mode'] == 'raw'][0]
    rff_m = [r for r in MLP if r['mode'] == 'rff_per_landmark' and not r['include_raw']][0]
    print(' [MLP]')
    compare(rff_m, raw_m, 'mlp_folds', 'RFF', 'raw')

# -------------------------------------------------------------- 停止规则
print(f"\n停止规则: 最好 CV-R² = {best_cv:+.4f} "
      f"{'>=' if best_cv >= STOP_RULE else '<'} {STOP_RULE} -> "
      f"{'可以继续' if best_cv >= STOP_RULE else '★ 停。不要加容量，去写负结果。'}")

# -------------------------------------------------------------- 报数口径
print(f'\n★ 报数必带: N=61 人（开发集 46）、P={P} 点、'
      'fold-local PCA、DC–15.85 kHz。')
print('★ SEM 被低估，不能当独立抽样误差用：')
print(f'   · {CFG["N_FOLDS"]} 折之间共享 {100*(CFG["N_FOLDS"]-1)/CFG["N_FOLDS"]:.0f}% 的训练数据；')
print(f'   · MLP 的 {CFG["N_SEEDS"]} 个种子跑在同一批 {len(DEV)} 个人身上。')
print('   论文里写成「CV 内部离散度」，不要写成标准误。')
if MLP:
    print(f'★ MLP 的 M/σ 由 ridge 选定，未针对 MLP 重调（§8 为公平特意固定）。')


In [ ]:
# ==========================================================================
# Cell 6b - 模型定义（端到端要用；与 FFM_2_3 / 2_6 / 2_8 逐字相同）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 7 — 端到端：ẑ 过冻结 decoder -> LSD，和群体均值基线比
#
# ★ 口径: 93 bins, DC–15.85 kHz, 44.1 kHz, dB magnitude,
#   LSD = 先沿频率 RMS，再对方向和样本【算术】平均（文献口径）
# ★ 这个数不能和 AE 重建的数放同一张表 —— 不同任务
# ==========================================================================
RUN_END2END = False        # 要出端到端数字时改 True

if not RUN_END2END:
    print('RUN_END2END = False。需要加载 FFM26_best.pth 和 SYMARE 的 .npy。')
    print('改成 True 之前先确认 Cell 6 的结论已经定稿。')
else:
    assert_Z_clean()          # ★ test 只跑一次，先确认 Z_SUB 干净
    import healpy as hp, zipfile
    # ---- 载入 AE（模型定义从 FFM_2_8 复制，或在此 exec）----
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(f'{SAVE_DIR}/FFM26_best.pth', map_location=device, weights_only=False)
    model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=LATENT_DIM,
                                 freq_embed_dim=32, num_datasets=4,
                                 num_bins=n_bins, tie_encoder_film=True).to(device)
    model.load_state_dict(ck['model_state_dict']); model.eval(); del ck

    def lsd(pred, gt):
        return float(np.sqrt(((pred - gt) ** 2).mean(1)).mean())

    # ---- 用最好的配置，在开发集上拟合，预测 test ----
    X_all = build_X(BEST['mode'], BEST['M'], BEST['sigma'], CFG['SEED'], BEST['include_raw'])
    fwd, inv, K = make_target(DEV)                       # ★ 只用开发集
    Xd, (Xt_,) = standardize(X_all[DEV], [X_all[I_TE]])
    m = Ridge(alpha=BEST.get('alpha', 1.0)).fit(Xd, fwd(Z_SUB[DEV]))
    Z_hat = inv(m.predict(Xt_))                          # (n_test, 5952)
    print(f'ẑ {Z_hat.shape} -> 每人 reshape ({n_bins}, {LATENT_DIM})')

    # ---- GT、预测、群体均值 ----
    # （GT 与 mean 需要 SYMARE 的 .npy —— 路径与 FFM_2_8 相同）
    print('★ 这一格需要 SYMARE 的 .npy 目录，接 FFM_2_8 的 Cell 0 使用。')
    print('  预测 LSD / 群体均值 LSD / 差值 三个数报出来，并注明口径。')

## 报数纪律

1. **判据不是「RFF 的差值为正」**，而是**差值大于两者 SEM 之和**。N=46 的开发集上，
   0.02 的差值毫无意义。

2. **停止规则 CV-R² < 0.10 是事先约定的**，不是事后找的。触发了就写负结果，
   不要加容量、不要加 epoch。

3. **每个数带作用域**：N=61（开发集 46 / test 15）、P=19 点、fold-local PCA、
   DC–15.85 kHz、`FFM26_best.pth`。

4. **三个数不能同框**：AE 重建（HRTF 进 HRTF 出）、端到端（几何进 HRTF 出）、
   latent 空间 R²。不同任务。

5. **test 只跑一次。** 15 个人，跑完就不要再改上游。

## 四种可能的解释，实验要能区分

1. raw XYZ 表示不够，RFF 有帮助
2. RFF 没帮助，几何→latent 的关系不是这种编码能抓的
3. MLP 只是在 N 太小的情况下过拟合
4. 稀疏 landmark 本身就不含足够信息

**如果 train R² ≈ 1 而 CV-R² ≈ 0 —— 那是 (3)，不要用加容量去回应。**

## 上游已知（2_8，正确配对）

dCor²ᵤ = 0.0710 (p = 0.0020)，负对照 −0.008 / −0.031。
分带 **DC–0.5k 0.140 / 0.5–6k 0.128 / 6–12k 0.063 / 12–15.8k 0.073** ——
信号最强在头尺寸主导的低频，**最弱恰好在 6–12 kHz 耳廓带**，而那正是 AE 重建最好
（残差/基线 25.3%）、个体化最需要的频段。